# Recombination

`xftsim` provides recombination machinery for both simplistic
constant-rate maps and (eventually) realistic empirical maps. In
practice we've observed that both procedures tend to yield very
similar results.


In [ ]:
import xftsim as xft
import numpy as np
from xftsim.reproduce import RecombinationMap


A `RecombinationMap` maps an ordered collection of `m` diploid
variants per chromosome to a vector of `m` probabilities `p`. The
first element of `p` in each chromosome is always 0.5 (Mendelian
inheritance); the rest reflect independent recombination probabilities
between contiguous loci. There are currently no mechanisms for
interference or population-specific recombination maps.

## Simple recombination schemes

The simplest scheme sets all elements of `p` (except the
chromosome-boundary entries) to a single value — 0.5 corresponds to
fully unlinked loci, and the strength of local LD increases as `p` →
0. Pass a single float for `p` to get this behaviour:


In [ ]:
hap = xft.founders.founder_haplotypes_uniform_AFs(n=200, m=80)
rmap = RecombinationMap(p=0.1, vid=hap.vid, chrom=hap.variants.chrom)
rmap


The same map can be built directly from a haplotype array using the
`from_haplotypes` staticmethod (which used to be called
`constant_map_from_haplotypes`):


In [ ]:
RecombinationMap.from_haplotypes(hap, p=0.1)

For a fully arbitrary per-locus map, pass a vector of probabilities
to `p` instead of a scalar:


In [ ]:
RecombinationMap(
    p=np.random.uniform(size=hap.m),
    vid=hap.vid,
    chrom=hap.variants.chrom,
)


## Realistic recombination schemes

The legacy `xftsim.struct.GeneticMap.from_pyrho_maps()` loader has not
yet been ported to the v0.9 `GeneticMap` class, and there is no
v0.9 helper for building a variable-rate `RecombinationMap` from
empirical cM distances on a haplotype array. We will document this
section once those helpers land. In the meantime, you can build a
custom variable map by passing a precomputed `p` vector to
`RecombinationMap` directly.

## GRG-native recombination

When founder haplotypes are loaded as a `GraphHaplotypeOperator`
(i.e., GRG-backed), `GraphHaplotypeOperator.meiosis()` performs
recombination directly on the GRG using a **bubble-insertion
(node-insertion) algorithm** implemented in
`xftsim.grg_recombination`. The per-locus crossover probabilities
from the `RecombinationMap` are translated into bp-space segments,
and offspring sample nodes are inserted into the graph without
materializing a dense genotype matrix. This means offspring remain
`GraphHaplotypeOperator` across all generations.

The `RecombinationMap` also supports an optional `pos_bp` parameter
that forces `p=0` between adjacent variants at the same base-pair
position. This is important for GRG-native meiosis because the
segment-based algorithm groups same-position variants into a single
segment; a crossover between them would produce incorrect haplotype
assignments. `RecombinationMap.from_haplotypes()` threads `pos_bp`
automatically when the haplotype array provides it.

An optional C++ backend (`xftsim.grg_recombination_native`) is
available for performance-critical applications. It is tried
automatically if installed, with transparent fallback to the Python
implementation. See the
[GRG-Backed Genotypes example](../examples/07_grg_genotypes.ipynb)
for a full walkthrough.